# Phase 2: Credit Card Fraud Detection with XGBoost

This notebook implements the complete XGBoost phase of the Enterprise Adaptive Learning Laboratory.

It follows the same evaluation contract as Phase 1:

- Same ULB `creditcard.csv`
- Same chronological 60/20/20 split
- Validation data for model selection, early stopping, and threshold selection
- Test data used only after the model and threshold are fixed
- Permanent artifacts saved to Google Drive

The notebook trains four controlled XGBoost candidates, including unweighted and imbalance-aware models, selects the best candidate using validation PR-AUC, compares it with the latest Phase 1 Logistic Regression run when available, and generates a reproducible run package.

> **Research POC only. Do not use this notebook for live financial decisions.**


## Why XGBoost now?

Logistic Regression established a simple baseline. XGBoost is a strong tabular-learning benchmark that builds trees sequentially, with later trees correcting errors from earlier trees.

The central question is not whether XGBoost is more advanced. It is whether it improves fraud ranking, false-positive behavior, and review-capacity metrics under the same data and test discipline.


## Permanent storage

- **GitHub:** project code and notebooks
- **Google Drive:** dataset and experiment artifacts
- **Colab `/content`:** temporary clone and computation

Expected dataset: `MyDrive/enterprise-adaptive-learning-lab-data/raw/creditcard.csv`


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
# 2. Define paths, clone/update GitHub, and install dependencies
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, random, shutil, subprocess, sys, time

REPOSITORY_URL='https://github.com/deepak-malviyaa/enterprise-adaptive-learning-lab.git'
PROJECT_ROOT=Path('/content/enterprise-adaptive-learning-lab')
DRIVE_ROOT=Path('/content/drive/MyDrive/enterprise-adaptive-learning-lab-data')
DRIVE_DATASET=DRIVE_ROOT/'raw'/'creditcard.csv'
DRIVE_OUTPUTS=DRIVE_ROOT/'outputs'
DRIVE_RUNS=DRIVE_OUTPUTS/'runs'
PROJECT_DATASET=PROJECT_ROOT/'data'/'raw'/'creditcard.csv'

if (PROJECT_ROOT/'.git').exists():
    result=subprocess.run(['git','-C',str(PROJECT_ROOT),'pull','--ff-only'],text=True,capture_output=True)
elif PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
    result=subprocess.run(['git','clone',REPOSITORY_URL,str(PROJECT_ROOT)],text=True,capture_output=True)
else:
    result=subprocess.run(['git','clone',REPOSITORY_URL,str(PROJECT_ROOT)],text=True,capture_output=True)
print(result.stdout); print(result.stderr)
if result.returncode!=0: raise RuntimeError('Git operation failed. Check repository access.')

required=['README.md','pyproject.toml','configs','data','notebooks','src','tests']
missing=[x for x in required if not (PROJECT_ROOT/x).exists()]
if missing: raise FileNotFoundError(f'Incomplete clone: {missing}')

pyproject=PROJECT_ROOT/'pyproject.toml'
text=pyproject.read_text(encoding='utf-8').replace('>=3.10,<3.13','>=3.10,<3.14')
pyproject.write_text(text,encoding='utf-8')
os.chdir(PROJECT_ROOT)
for command in [
    [sys.executable,'-m','pip','install','-q','-e','.'],
    [sys.executable,'-m','pip','install','-q','xgboost>=2,<4'],
]:
    install=subprocess.run(command,text=True,capture_output=True)
    print(install.stderr)
    if install.returncode!=0: raise RuntimeError(f'Installation failed: {command}')
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
print('Project ready:',PROJECT_ROOT)


In [ ]:
# 3. Connect the permanent dataset to the project
if not DRIVE_DATASET.exists():
    raise FileNotFoundError(f'Dataset missing: {DRIVE_DATASET}')
PROJECT_DATASET.parent.mkdir(parents=True,exist_ok=True)
if PROJECT_DATASET.exists() or PROJECT_DATASET.is_symlink(): PROJECT_DATASET.unlink()
PROJECT_DATASET.symlink_to(DRIVE_DATASET)
DATA_PATH=PROJECT_DATASET
print('Dataset link:',DATA_PATH)
print('Permanent source:',DATA_PATH.resolve())
print('Size MB:',round(DRIVE_DATASET.stat().st_size/1024**2,2))


In [ ]:
# 4. Imports and reproducibility
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import yaml
import xgboost as xgb
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score,brier_score_loss,classification_report,confusion_matrix,f1_score,precision_recall_curve,precision_score,recall_score,roc_auc_score
from xgboost import XGBClassifier

SEED=42
os.environ['PYTHONHASHSEED']=str(SEED)
random.seed(SEED); np.random.seed(SEED)
print('Python:',sys.version.split()[0]); print('XGBoost:',xgb.__version__); print('Seed:',SEED)


## Dataset validation and identity

The SHA-256 checksum identifies the exact dataset file. Schema validation prevents training on an incorrect or corrupted CSV.


In [ ]:
# 5. Load, checksum, and validate the dataset
def sha256_file(path,block_size=1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(block_size),b''): digest.update(block)
    return digest.hexdigest()

DATASET_SHA256=sha256_file(DATA_PATH)
df=pd.read_csv(DATA_PATH)
EXPECTED_COLUMNS=['Time']+[f'V{i}' for i in range(1,29)]+['Amount','Class']
missing=sorted(set(EXPECTED_COLUMNS)-set(df.columns)); errors=[]
if df.empty: errors.append('Dataset is empty')
if missing: errors.append(f'Missing columns: {missing}')
if df.columns.duplicated().any(): errors.append('Duplicate column names')
if 'Class' in df and df['Class'].isna().any(): errors.append('Missing target values')
if 'Class' in df and set(df['Class'].dropna().unique())!={0,1}: errors.append('Class must contain 0 and 1')
for column in EXPECTED_COLUMNS:
    if column in df and not pd.api.types.is_numeric_dtype(df[column]): errors.append(f'{column} is not numeric')
if 'Amount' in df and (df['Amount']<0).any(): errors.append('Negative Amount exists')
if np.isinf(df.select_dtypes(include='number').to_numpy()).any(): errors.append('Infinite numeric value exists')
if errors: raise ValueError('; '.join(errors))
print('DATASET VALIDATION PASSED')
print('Rows:',f'{len(df):,}','Columns:',len(df.columns),'Frauds:',int(df.Class.sum()))
print('Fraud rate:',f'{df.Class.mean():.6%}'); print('SHA-256:',DATASET_SHA256)
display(df.head())


## Chronological 60/20/20 split

Earlier transactions train the model. The next period is used for early stopping, model selection, and threshold selection. The latest transactions form the untouched final test set.


In [ ]:
# 6. Build and verify chronological partitions
df_sorted=df.sort_values('Time',kind='mergesort').reset_index(drop=True)
df_sorted['row_id']=np.arange(len(df_sorted))
train_end=int(len(df_sorted)*0.60); val_end=int(len(df_sorted)*0.80)
train_df=df_sorted.iloc[:train_end].copy(); val_df=df_sorted.iloc[train_end:val_end].copy(); test_df=df_sorted.iloc[val_end:].copy()
assert train_df.Time.max()<=val_df.Time.min()<=test_df.Time.min()
assert set(train_df.row_id).isdisjoint(val_df.row_id)
assert set(train_df.row_id).isdisjoint(test_df.row_id)
assert set(val_df.row_id).isdisjoint(test_df.row_id)
for name,part in [('train',train_df),('validation',val_df),('test',test_df)]:
    if part.Class.nunique()!=2: raise ValueError(f'{name} does not contain both classes')
partition_summary=pd.DataFrame([
 {'partition':name,'rows':len(part),'frauds':int(part.Class.sum()),'fraud_rate':float(part.Class.mean()),'minimum_time':float(part.Time.min()),'maximum_time':float(part.Time.max()),'minimum_row_id':int(part.row_id.min()),'maximum_row_id':int(part.row_id.max())}
 for name,part in [('train',train_df),('validation',val_df),('test',test_df)]
])
display(partition_summary)


In [ ]:
# 7. Create feature matrices and calculate class imbalance weight
DROP_COLUMNS=['Class','row_id']
X_train=train_df.drop(columns=DROP_COLUMNS); y_train=train_df.Class
X_val=val_df.drop(columns=DROP_COLUMNS); y_val=val_df.Class
X_test=test_df.drop(columns=DROP_COLUMNS); y_test=test_df.Class
negative_count=int((y_train==0).sum()); positive_count=int((y_train==1).sum())
SCALE_POS_WEIGHT=negative_count/positive_count
print('Train:',X_train.shape,'Validation:',X_val.shape,'Test:',X_test.shape)
print('scale_pos_weight:',round(SCALE_POS_WEIGHT,4))


## Load Phase 1 results if available

The latest successful Logistic Regression run is loaded for direct comparison. XGBoost can still run when no Phase 1 artifacts are found.


In [ ]:
# 8. Find latest Phase 1 run and verify basic split compatibility
DRIVE_RUNS.mkdir(parents=True,exist_ok=True)
phase1_runs=sorted([p for p in DRIVE_RUNS.glob('*_phase1_fraud_logistic') if (p/'test_metrics.json').exists()],reverse=True)
PHASE1_RUN=phase1_runs[0] if phase1_runs else None
phase1_metrics=None; phase1_review=None
if PHASE1_RUN:
    phase1_metrics=json.loads((PHASE1_RUN/'test_metrics.json').read_text(encoding='utf-8'))
    if (PHASE1_RUN/'review_capacity.csv').exists(): phase1_review=pd.read_csv(PHASE1_RUN/'review_capacity.csv')
    if (PHASE1_RUN/'partition_summary.csv').exists():
        old_split=pd.read_csv(PHASE1_RUN/'partition_summary.csv')[['partition','rows','frauds']].reset_index(drop=True)
        new_split=partition_summary[['partition','rows','frauds']].reset_index(drop=True)
        if not old_split.equals(new_split): raise ValueError('Phase 1 and Phase 2 split summaries differ')
    print('Loaded Phase 1:',PHASE1_RUN)
else:
    print('No Phase 1 artifacts found; comparison will contain XGBoost only.')


## Controlled XGBoost search

Four candidates are tested. `aucpr` is monitored on validation data, and early stopping selects the useful number of trees. `scale_pos_weight` is calculated from the training partition only.


In [ ]:
# 9. Define candidate configurations
COMMON={'objective':'binary:logistic','eval_metric':'aucpr','tree_method':'hist','n_estimators':2500,'early_stopping_rounds':75,'random_state':SEED,'n_jobs':-1,'verbosity':0}
CANDIDATES=[
 {'candidate_name':'basic_depth4','learning_rate':0.05,'max_depth':4,'min_child_weight':1,'subsample':0.90,'colsample_bytree':0.90,'reg_lambda':1.0,'reg_alpha':0.0,'scale_pos_weight':1.0},
 {'candidate_name':'weighted_depth4','learning_rate':0.05,'max_depth':4,'min_child_weight':1,'subsample':0.90,'colsample_bytree':0.90,'reg_lambda':1.0,'reg_alpha':0.0,'scale_pos_weight':SCALE_POS_WEIGHT},
 {'candidate_name':'weighted_depth6_regularized','learning_rate':0.03,'max_depth':6,'min_child_weight':5,'subsample':0.85,'colsample_bytree':0.85,'reg_lambda':5.0,'reg_alpha':0.10,'scale_pos_weight':SCALE_POS_WEIGHT},
 {'candidate_name':'weighted_shallow_conservative','learning_rate':0.03,'max_depth':3,'min_child_weight':3,'subsample':0.80,'colsample_bytree':0.90,'reg_lambda':10.0,'reg_alpha':0.25,'scale_pos_weight':SCALE_POS_WEIGHT},
]
display(pd.DataFrame(CANDIDATES))


In [ ]:
# 10. Reusable metric functions
def ranking_metrics(y_true,prob):
    return {'pr_auc':float(average_precision_score(y_true,prob)),'roc_auc':float(roc_auc_score(y_true,prob)),'brier_score':float(brier_score_loss(y_true,prob))}

def threshold_table(y_true,prob):
    rows=[]
    for threshold in np.round(np.arange(0.01,1.00,0.01),2):
        pred=(prob>=threshold).astype(int); tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
        rows.append({'threshold':float(threshold),'precision':float(precision_score(y_true,pred,zero_division=0)),'recall':float(recall_score(y_true,pred,zero_division=0)),'f1':float(f1_score(y_true,pred,zero_division=0)),'alerts':int(pred.sum()),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)})
    return pd.DataFrame(rows)

def at_k(y_true,prob,k):
    ranked=pd.DataFrame({'actual':np.asarray(y_true),'probability':np.asarray(prob)}).sort_values('probability',ascending=False)
    selected=ranked.head(min(k,len(ranked))); found=int(selected.actual.sum()); total=int(ranked.actual.sum())
    return {'k':int(k),'frauds_found':found,'precision_at_k':float(found/len(selected)),'recall_at_k':float(found/total if total else 0.0)}


In [ ]:
# 11. Train candidates with validation early stopping
candidate_models={}; search_rows=[]
for candidate in CANDIDATES:
    name=candidate['candidate_name']; params={**COMMON,**{k:v for k,v in candidate.items() if k!='candidate_name'}}
    model=XGBClassifier(**params); started=time.perf_counter()
    model.fit(X_train,y_train,eval_set=[(X_train,y_train),(X_val,y_val)],verbose=False)
    seconds=time.perf_counter()-started; val_prob=model.predict_proba(X_val)[:,1]; metrics=ranking_metrics(y_val,val_prob)
    candidate_models[name]={'model':model,'val_prob':val_prob,'parameters':params}
    search_rows.append({'candidate_name':name,**metrics,'best_iteration':int(model.best_iteration),'best_xgboost_aucpr':float(model.best_score),'training_seconds':float(seconds),**{k:v for k,v in candidate.items() if k!='candidate_name'}})
    print(name,'PR-AUC',round(metrics['pr_auc'],6),'best iteration',model.best_iteration,'seconds',round(seconds,2))
parameter_search=pd.DataFrame(search_rows).sort_values(['pr_auc','brier_score'],ascending=[False,True]).reset_index(drop=True)
display(parameter_search)


In [ ]:
# 12. Select final candidate using validation PR-AUC
BEST_NAME=str(parameter_search.iloc[0].candidate_name)
BEST_INFO=candidate_models[BEST_NAME]; BEST_MODEL=BEST_INFO['model']; BEST_VAL_PROB=BEST_INFO['val_prob']; BEST_PARAMETERS=BEST_INFO['parameters']
print('Selected:',BEST_NAME); print('Best iteration:',BEST_MODEL.best_iteration)
print(json.dumps(BEST_PARAMETERS,indent=2,default=float))

history=BEST_MODEL.evals_result(); train_aucpr=history['validation_0']['aucpr']; val_aucpr=history['validation_1']['aucpr']; rounds=np.arange(1,len(train_aucpr)+1)
plt.figure(figsize=(9,5)); plt.plot(rounds,train_aucpr,label='Training AUCPR'); plt.plot(rounds,val_aucpr,label='Validation AUCPR'); plt.axvline(BEST_MODEL.best_iteration+1,color='black',linestyle='--',label='Best iteration'); plt.xlabel('Boosting round'); plt.ylabel('AUCPR'); plt.title('XGBoost learning curve'); plt.legend(); plt.tight_layout(); plt.show()


## Validation threshold selection

The model is selected first. Then validation probabilities select the threshold with maximum F1. Test data is still untouched.


In [ ]:
# 13. Select threshold using validation data only
validation_thresholds=threshold_table(y_val,BEST_VAL_PROB)
max_f1=validation_thresholds.f1.max()
selected=validation_thresholds[validation_thresholds.f1==max_f1].sort_values(['recall','threshold'],ascending=[False,True]).iloc[0]
SELECTED_THRESHOLD=float(selected.threshold)
print('Selected threshold:',SELECTED_THRESHOLD)
display(validation_thresholds.sort_values('f1',ascending=False).head(10))

precision_curve,recall_curve,_=precision_recall_curve(y_val,BEST_VAL_PROB)
fig,axes=plt.subplots(1,2,figsize=(15,5))
axes[0].plot(recall_curve,precision_curve); axes[0].axhline(y_val.mean(),linestyle='--'); axes[0].set(xlabel='Recall',ylabel='Precision',title='Validation precision-recall curve')
axes[1].plot(validation_thresholds.threshold,validation_thresholds.precision,label='Precision'); axes[1].plot(validation_thresholds.threshold,validation_thresholds.recall,label='Recall'); axes[1].plot(validation_thresholds.threshold,validation_thresholds.f1,label='F1'); axes[1].axvline(SELECTED_THRESHOLD,color='black',linestyle='--'); axes[1].set(xlabel='Threshold',ylabel='Metric',title='Threshold trade-off'); axes[1].legend(); plt.tight_layout(); plt.show()


## Final test evaluation

The selected candidate and threshold are now fixed. The final test result includes ranking metrics, threshold metrics, inference time, confusion matrix, and review-capacity performance.


In [ ]:
# 14. Evaluate once on the untouched test set
started=time.perf_counter(); test_prob=BEST_MODEL.predict_proba(X_test)[:,1]; prediction_seconds=time.perf_counter()-started
test_pred=(test_prob>=SELECTED_THRESHOLD).astype(int); tn,fp,fn,tp=confusion_matrix(y_test,test_pred,labels=[0,1]).ravel()
final_metrics={**ranking_metrics(y_test,test_prob),'candidate_name':BEST_NAME,'threshold':SELECTED_THRESHOLD,'precision':float(precision_score(y_test,test_pred,zero_division=0)),'recall':float(recall_score(y_test,test_pred,zero_division=0)),'f1':float(f1_score(y_test,test_pred,zero_division=0)),'alerts':int(test_pred.sum()),'true_negatives':int(tn),'false_positives':int(fp),'false_negatives':int(fn),'true_positives':int(tp),'best_iteration':int(BEST_MODEL.best_iteration),'training_seconds':float(parameter_search.iloc[0].training_seconds),'test_prediction_seconds':float(prediction_seconds),'test_microseconds_per_row':float(prediction_seconds/len(X_test)*1_000_000)}
print(json.dumps(final_metrics,indent=2)); print('Detected frauds:',tp); print('Missed frauds:',fn); print('Legitimate transactions incorrectly flagged:',fp); print(classification_report(y_test,test_pred,digits=4,zero_division=0))

confusion=np.array([[tn,fp],[fn,tp]]); plt.figure(figsize=(6,5)); plt.imshow(confusion,cmap='Blues')
for r in range(2):
    for c in range(2): plt.text(c,r,f'{confusion[r,c]:,}',ha='center',va='center')
plt.xticks([0,1],['Predicted legitimate','Predicted fraud']); plt.yticks([0,1],['Actual legitimate','Actual fraud']); plt.title('XGBoost test confusion matrix'); plt.colorbar(); plt.tight_layout(); plt.show()

review_capacity=pd.DataFrame([at_k(y_test,test_prob,k) for k in [50,100,250,500]])
display(review_capacity)


## Feature importance

Gain importance describes how much a feature improves tree splits. Permutation importance estimates the validation PR-AUC reduction when a feature is shuffled. Neither proves causality, and anonymized `V` features should not be given invented business meanings.


In [ ]:
# 15. Gain and permutation importance
booster=BEST_MODEL.get_booster(); gain=booster.get_score(importance_type='gain')
feature_importance=pd.DataFrame({'feature':X_train.columns,'gain':[float(gain.get(f,0.0)) for f in X_train.columns]}).sort_values('gain',ascending=False).reset_index(drop=True)
feature_importance['gain_fraction']=feature_importance.gain/feature_importance.gain.sum()
display(feature_importance.head(15))

sample_size=min(20000,len(X_val)); sample_index=X_val.sample(n=sample_size,random_state=SEED).index
permutation=permutation_importance(BEST_MODEL,X_val.loc[sample_index],y_val.loc[sample_index],scoring='average_precision',n_repeats=5,random_state=SEED,n_jobs=-1)
permutation_table=pd.DataFrame({'feature':X_train.columns,'importance_mean':permutation.importances_mean,'importance_std':permutation.importances_std}).sort_values('importance_mean',ascending=False).reset_index(drop=True)
display(permutation_table.head(15))


## Phase 1 comparison

If Phase 1 artifacts exist, this section compares Logistic Regression and XGBoost. This is a descriptive comparison; stronger statistical claims require confidence intervals or repeated resampling.


In [ ]:
# 16. Compare final models
rows=[]
if phase1_metrics:
    rows.append({'model':'phase1_logistic_regression',**{k:phase1_metrics.get(k) for k in ['pr_auc','roc_auc','precision','recall','f1','brier_score','false_positives','false_negatives','training_seconds']}})
rows.append({'model':f'phase2_xgboost_{BEST_NAME}',**{k:final_metrics.get(k) for k in ['pr_auc','roc_auc','precision','recall','f1','brier_score','false_positives','false_negatives','training_seconds']}})
model_comparison=pd.DataFrame(rows).set_index('model'); display(model_comparison)
if phase1_review is not None:
    review_comparison=phase1_review.merge(review_capacity,on='k',how='outer',suffixes=('_logistic','_xgboost')); display(review_comparison)
else: review_comparison=review_capacity.copy()


## Save the complete experiment

The run package includes model JSON, checksum, split manifest, candidate search, threshold search, predictions, feature importance, model comparison, figures, and a Markdown report.


In [ ]:
# 17. Create permanent Phase 2 run folder and save artifacts
for name in ['runs','models','metrics','predictions','figures','reports']: (DRIVE_OUTPUTS/name).mkdir(parents=True,exist_ok=True)
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_phase2_fraud_xgboost'; RUN_DIR=DRIVE_RUNS/RUN_ID; FIGURE_DIR=RUN_DIR/'figures'; RUN_DIR.mkdir(parents=True); FIGURE_DIR.mkdir()

configuration={'phase':'phase2_fraud_xgboost','seed':SEED,'dataset_path':str(DRIVE_DATASET),'dataset_sha256':DATASET_SHA256,'split':{'strategy':'chronological','train':0.60,'validation':0.20,'test':0.20},'model_selection_metric':'validation_pr_auc','threshold_selection':'maximum validation F1','selected_candidate':BEST_NAME,'selected_threshold':SELECTED_THRESHOLD,'selected_parameters':BEST_PARAMETERS,'review_capacity':[50,100,250,500]}
environment={'rows':len(df),'columns':len(df.columns),'frauds':int(df.Class.sum()),'fraud_rate':float(df.Class.mean()),'duplicates':int(df.duplicated().sum()),'python':sys.version,'pandas':pd.__version__,'numpy':np.__version__,'sklearn':sklearn.__version__,'xgboost':xgb.__version__,'platform':platform.platform()}
split_manifest={'dataset_sha256':DATASET_SHA256,'total_rows':len(df_sorted),'train':{'min_row_id':int(train_df.row_id.min()),'max_row_id':int(train_df.row_id.max()),'rows':len(train_df)},'validation':{'min_row_id':int(val_df.row_id.min()),'max_row_id':int(val_df.row_id.max()),'rows':len(val_df)},'test':{'min_row_id':int(test_df.row_id.min()),'max_row_id':int(test_df.row_id.max()),'rows':len(test_df)}}
predictions=pd.DataFrame({'row_id':test_df.row_id,'Time':test_df.Time,'Amount':test_df.Amount,'actual_class':y_test,'fraud_probability':test_prob,'predicted_class':test_pred,'selected_threshold':SELECTED_THRESHOLD})

BEST_MODEL.save_model(RUN_DIR/'xgboost_model.json')
(RUN_DIR/'config.yaml').write_text(yaml.safe_dump(configuration,sort_keys=False),encoding='utf-8')
(RUN_DIR/'dataset_and_environment.json').write_text(json.dumps(environment,indent=2),encoding='utf-8')
(RUN_DIR/'split_manifest.json').write_text(json.dumps(split_manifest,indent=2),encoding='utf-8')
(RUN_DIR/'test_metrics.json').write_text(json.dumps(final_metrics,indent=2),encoding='utf-8')
parameter_search.to_csv(RUN_DIR/'parameter_search.csv',index=False); partition_summary.to_csv(RUN_DIR/'partition_summary.csv',index=False); validation_thresholds.to_csv(RUN_DIR/'validation_thresholds.csv',index=False); review_capacity.to_csv(RUN_DIR/'review_capacity.csv',index=False); feature_importance.to_csv(RUN_DIR/'feature_importance_gain.csv',index=False); permutation_table.to_csv(RUN_DIR/'feature_importance_permutation.csv',index=False); model_comparison.to_csv(RUN_DIR/'model_comparison.csv'); review_comparison.to_csv(RUN_DIR/'review_capacity_comparison.csv',index=False); predictions.to_csv(RUN_DIR/'test_predictions.csv',index=False)

report_lines=['# Phase 2 Fraud Detection Report: XGBoost','','## Executive summary',f'Selected candidate: `{BEST_NAME}` using validation PR-AUC. Early stopping monitored validation AUCPR. The threshold was selected using validation F1 before final test evaluation.','','## Dataset',f'- Rows: {len(df):,}',f'- Frauds: {int(df.Class.sum()):,}',f'- Fraud rate: {df.Class.mean():.6%}',f'- SHA-256: `{DATASET_SHA256}`','','## Final test results',f'- PR-AUC: {final_metrics["pr_auc"]:.6f}',f'- ROC-AUC: {final_metrics["roc_auc"]:.6f}',f'- Precision: {final_metrics["precision"]:.6f}',f'- Recall: {final_metrics["recall"]:.6f}',f'- F1: {final_metrics["f1"]:.6f}',f'- Brier score: {final_metrics["brier_score"]:.6f}',f'- Detected frauds: {tp}',f'- Missed frauds: {fn}',f'- Legitimate transactions incorrectly flagged: {fp}','','## Limitations','- Research POC only.','- V1-V28 are anonymized PCA-derived features.','- Time is relative, not a production timestamp.','- Maximum validation F1 is not a validated banking-cost policy.','- The small controlled search does not guarantee a global optimum.','','## Next phase','Implement Deep MLP using the same checksum, split manifest, and test discipline.']
(RUN_DIR/'report.md').write_text('\n'.join(report_lines)+'\n',encoding='utf-8')

# Save key figures
plt.figure(figsize=(9,5)); plt.plot(rounds,train_aucpr,label='Training'); plt.plot(rounds,val_aucpr,label='Validation'); plt.axvline(BEST_MODEL.best_iteration+1,color='black',linestyle='--'); plt.xlabel('Boosting round'); plt.ylabel('AUCPR'); plt.legend(); plt.tight_layout(); plt.savefig(FIGURE_DIR/'learning_curve.png',dpi=150); plt.close()
plt.figure(figsize=(9,6)); top=feature_importance.head(15).sort_values('gain'); plt.barh(top.feature,top.gain); plt.xlabel('Average gain'); plt.tight_layout(); plt.savefig(FIGURE_DIR/'feature_importance_gain.png',dpi=150); plt.close()
archive=Path(shutil.make_archive(str(RUN_DIR),'zip',root_dir=RUN_DIR))
print('Artifacts:',RUN_DIR); print('ZIP:',archive)


In [ ]:
# 18. Phase 2 acceptance checks
checks={'dataset_exists':DATA_PATH.exists(),'schema_valid':set(EXPECTED_COLUMNS).issubset(df.columns),'checksum_recorded':bool(DATASET_SHA256),'chronological_order':train_df.Time.max()<=val_df.Time.min()<=test_df.Time.min(),'partitions_disjoint':set(train_df.row_id).isdisjoint(val_df.row_id) and set(train_df.row_id).isdisjoint(test_df.row_id) and set(val_df.row_id).isdisjoint(test_df.row_id),'all_candidates_completed':len(parameter_search)==len(CANDIDATES),'early_stopping_completed':hasattr(BEST_MODEL,'best_iteration'),'threshold_valid':0<SELECTED_THRESHOLD<1,'model_saved':(RUN_DIR/'xgboost_model.json').exists(),'metrics_saved':(RUN_DIR/'test_metrics.json').exists(),'predictions_saved':(RUN_DIR/'test_predictions.csv').exists(),'report_saved':(RUN_DIR/'report.md').exists()}
display(pd.Series(checks,name='passed').to_frame())
failed=[k for k,v in checks.items() if not v]
if failed: raise AssertionError(f'Acceptance checks failed: {failed}')
print('PHASE 2 XGBOOST COMPLETED SUCCESSFULLY')
print('Selected candidate:',BEST_NAME); print('Run folder:',RUN_DIR)


## Review before the next phase

Inspect `report.md`, `model_comparison.csv`, `test_metrics.json`, `review_capacity.csv`, `parameter_search.csv`, and both feature-importance files.

Only after reviewing false positives, false negatives, threshold behavior, PR-AUC, and review-capacity results should the Deep MLP notebook be generated.


## Save this notebook to GitHub

After success, choose **File → Save a copy in GitHub**:

- Repository: `deepak-malviyaa/enterprise-adaptive-learning-lab`
- Branch: `main`
- Path: `notebooks/11_bfsi_fraud_xgboost.ipynb`
- Commit: `Add Phase 2 XGBoost fraud notebook`

Do not commit datasets, tokens, models, predictions, or Drive credentials.
